# G05: Diagnostic

*Which pairs are hard, and is difficulty a property of the pair or of the model?*

| Sub-metric | Definition | Quantification | Scoring |
|---|---|---|---|
| **Per-pair difficulty** | mean reciprocal rank of the positive on the KG arms, per pair | pooled over models, seeds and shuffles; `mrr_grounded` restricts to arms whose graph has evidence for the pair | reported |
| **Cross-model agreement** | correlation of per-pair MRR between models | Pearson matrix and its mean off-diagonal | high agreement means the difficulty is in the pair |
| **Drivers** | therapeutic area, number of graphs with evidence for the pair, the model's confidence and its reported evidence agreement | correlations and group means | reported |

**Reported, not scored.** Everything here; this stage explains the G03 result rather than adding to it.

**Inputs:** `results/tables/G02_run/runs/` · `results/checkpoints/G01_benchmark.pkl` · `config/params.yaml` (`grounding`)

**Outputs:** `results/{figures,tables}/G05_diagnostic/` · `results/checkpoints/G05_diagnostic.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

In [1]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import gc
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

from IPython.display import display
import numpy as np
import pandas as pd


from scipy import stats

from biokgsuite.grounding import cross_model_corr, difficulty_table, load_runs, mean_pairwise_corr, validate_runs
from biokgsuite.viz.grounding import (fig_coverage_fair, fig_difficulty_by_area, fig_difficulty_drivers,
                                      fig_hardest_pairs, fig_intrinsic_difficulty)
from biokgsuite.io import Outputs, find_config, load_config, require
from biokgsuite.viz.style import setup_style

setup_style()

In [2]:
# Config, parameters and the results writer.
config = load_config(find_config(_root), trial_subset=False)   # graphs are named below, not by the panel trial file
params = config['analysis_params']
G      = require(params, 'grounding')
BASE   = config['_base_dir']

# Every figure, table and checkpoint goes through `out`, as in notebooks 01-07.
out = Outputs('G05_diagnostic', BASE)

KGS       = require(G, 'kgs')
MODELS    = require(G, 'models')
LABELS    = require(G, 'model_labels') | {'none': 'No KG', 'primekg': 'PrimeKG', 'drkg': 'DRKG',
                                           'biokg': 'BioKG', 'openbilink': 'OpenBioLink', 'hetionet': 'Hetionet'}
FAMILIES  = require(G, 'families')
POOL_SIZE = require(G, 'pool_size')
SHUFFLES  = require(G, 'shuffles')
SEEDS     = require(G, 'seeds')
STRICT    = require(G, 'strict')
N_BOOT    = require(G, 'n_bootstrap')
SEED      = require(params, 'random_seed')
RNG       = np.random.default_rng(SEED)
# BKG_MODE=mock reads the rehearsal runs 02 wrote under runs_mock/; real
# (the default) reads the results and refuses any mock row it finds there.
MOCK      = os.environ.get('BKG_MODE', 'real') == 'mock'
RUN_DIR   = Path(BASE) / 'results' / 'tables' / 'G02_run' / ('runs_mock' if MOCK else 'runs')

### Loading

In [3]:
bench = out.load_checkpoint('G01_benchmark')['benchmark']
runs = validate_runs(load_runs(RUN_DIR, families=['main'], models=MODELS, allow_mock=MOCK))
print(f'{len(runs)} rows, {runs.pair_id.nunique()} pairs, models {sorted(runs.model.unique())}')

13824 rows, 128 pairs, models ['gemini:gemini-3.5-flash-lite', 'gpt-4.1-mini', 'llama3.3:70b']


## 1. Per-pair difficulty

In [4]:
per = difficulty_table(runs, bench)
print(f'{len(per)} pairs; mean MRR on the KG arms {per.mrr.mean():.3f}, on grounded arms only {per.mrr_grounded.mean():.3f}')
display(per[['disease_name', 'drug_name', 'area', 'modality', 'mrr', 'mrr_none', 'hit1', 'n_kg_grounded']].head(15).round(3))
out.table(per, 'difficulty')

128 pairs; mean MRR on the KG arms 0.632, on grounded arms only 0.806


,disease_name,drug_name,area,modality,mrr,mrr_none,hit1,n_kg_grounded
0,chronic rhinosinusitis,tezepelumab,Immunology,antibody,0.155,0.409,0.000,0
1,Crohn disease,mirikizumab,Immunology,unknown,0.166,0.275,0.012,0
2,psoriatic arthritis,bimekizumab,Immunology,antibody,0.174,0.307,0.012,0
3,anal squamous cell carcinoma,retifanlimab,Oncology,unknown,0.181,0.392,0.012,0
4,obstructive sleep apnea syndrome,tirzepatide,Respiratory,protein,0.186,0.273,0.012,0
5,complement 3 glomerulopathy,pegcetacoplan,Immunology,unknown,0.191,0.264,0.012,0
6,endometriosis,linzagolix,unclassified,unknown,0.191,0.400,0.025,0
7,heart failure,finerenone,Cardiovascular,small molecule,0.193,0.275,0.000,0
8,hidradenitis suppurativa,bimekizumab,Immunology,antibody,0.193,0.343,0.012,0
9,prurigo nodularis,nemolizumab,unclassified,antibody,0.196,0.249,0.012,0


PosixPath('/home/moline/biokgsuite-bib/results/tables/G05_diagnostic/difficulty.csv')

## 2. Cross-model agreement

In [5]:
corr = cross_model_corr(per)
corr.index = [LABELS.get(m, m) for m in corr.index]; corr.columns = [LABELS.get(m, m) for m in corr.columns]
display(corr.round(3))
print(f'mean pairwise correlation of per-pair MRR across models: {mean_pairwise_corr(corr):.3f}')
out.table(corr.reset_index().rename(columns={'index': 'model'}), 'cross_model_corr')

,Gemini-3.5-Flash-Lite,GPT-4.1-mini,Llama-3.3-70B
Gemini-3.5-Flash-Lite,1.000,0.959,0.911
GPT-4.1-mini,0.959,1.000,0.939
Llama-3.3-70B,0.911,0.939,1.000


mean pairwise correlation of per-pair MRR across models: 0.936


PosixPath('/home/moline/biokgsuite-bib/results/tables/G05_diagnostic/cross_model_corr.csv')

## 3. What predicts difficulty

Hard and easy tertiles, the correlation of per-pair MRR with each covariate, and the mean by therapeutic area.

In [6]:
q1, q2 = per.mrr.quantile([1 / 3, 2 / 3])
per['tertile'] = np.where(per.mrr <= q1, 'hard', np.where(per.mrr >= q2, 'easy', 'middle'))
display(per.groupby('tertile')[['mrr', 'pos_confidence', 'frac_consistent', 'n_kg_grounded', 'n_kg_covered']].mean().round(3))
drivers = pd.DataFrame([
    {'covariate': c, 'pearson_r': stats.pearsonr(per[c].astype(float), per.mrr)[0],
     'p': stats.pearsonr(per[c].astype(float), per.mrr)[1]}
    for c in ['n_kg_grounded', 'n_kg_covered', 'pos_confidence', 'frac_consistent', 'approval_year'] if per[c].notna().all()])
display(drivers.round(3))
by_area = per.groupby('area').agg(n=('mrr', 'size'), mrr=('mrr', 'mean')).sort_values('mrr')
display(by_area.round(3))
out.table(drivers, 'difficulty_drivers'); out.table(by_area.reset_index(), 'difficulty_by_area')

,mrr,pos_confidence,frac_consistent,n_kg_grounded,n_kg_covered
tertile,,,,,
easy,0.918,4.278,0.847,2.140,2.140
hard,0.276,1.335,0.071,0.395,0.930
middle,0.702,3.136,0.543,2.214,2.238


,covariate,pearson_r,p
0,n_kg_grounded,0.626,0.000
1,n_kg_covered,0.435,0.000
2,pos_confidence,0.972,0.000
3,frac_consistent,0.969,0.000
4,approval_year,0.007,0.941


,n,mrr
area,,
Infectious,4,0.396
Neurology,9,0.503
Metabolic,4,0.520
Respiratory,3,0.607
unclassified,14,0.618
Immunology,37,0.655
Cardiovascular,11,0.657
Oncology,46,0.668


PosixPath('/home/moline/biokgsuite-bib/results/tables/G05_diagnostic/difficulty_by_area.csv')

### Visualisation

In [7]:
out.figure(fig_intrinsic_difficulty(per, LABELS), 'G05a_cross_model');
out.figure(fig_hardest_pairs(per, n=20, labels=LABELS), 'G05b_hardest_pairs');
out.figure(fig_difficulty_drivers(per), 'G05c_drivers');
out.figure(fig_difficulty_by_area(per), 'G05d_by_area');
out.figure(fig_coverage_fair(per), 'G05e_coverage');

  saved G05a_cross_model.pdf and .png (600 dpi)


  saved G05b_hardest_pairs.pdf and .png (600 dpi)


  saved G05c_drivers.pdf and .png (600 dpi)


  saved G05d_by_area.pdf and .png (600 dpi)


  saved G05e_coverage.pdf and .png (600 dpi)


### Checkpoint

In [8]:
out.checkpoint({'difficulty': per, 'cross_model_corr': corr, 'mean_corr': mean_pairwise_corr(corr), 'drivers': drivers})

PosixPath('/home/moline/biokgsuite-bib/results/checkpoints/G05_diagnostic.pkl')

In [9]:
# Memory cleanup: free state before the next notebook in the pipeline.
for _v in ('kgs', 'graphs', 'dossiers', 'runs', 'bench'):
    globals().pop(_v, None)
gc.collect()
print('done:', out.notebook)

done: G05_diagnostic
